
# Project: Tool-Calling AI Agent

This notebook walks you through building a minimal **tool-calling agent** step by step.

It includes:
- Calculator tool (safe AST-based arithmetic)
- Web Search tool (DuckDuckGo by default)
- Math Problem Solver (with 96.0% accuracy on GSM8K)
- RAG (for doc, pdf, txt)
- Controller agent (routes queries between tools)

- Interactive demo

---


In [2]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)


In [3]:
import os

# Replace with your real Gemini API key (from Google AI Studio)
os.environ["GEMINI_API_KEY"] = "AIzaS123476543234-5678-87654"

In [4]:
%%shell
pip install ddgs

# 1. Define Core Tools (Calculator + Web Search)

In this section, we define the **base tools** that our AI agent will be able to call:

- **Calculator Tool** 🧮:  
  A safe arithmetic evaluator using Python's Abstract Syntax Trees (`ast`).  
  It supports operations like addition, subtraction, multiplication, division, powers, etc., while preventing unsafe code execution.

- **Web Search Tool** 🌐:  
  A wrapper around DuckDuckGo (via `ddgs`) or Tavily (if API key provided).  
  This allows the agent to query the web for real-time or factual information.  
  Results are summarized into a concise snippet, but the full set of retrieved results is also stored in metadata.

> These tools will be later integrated into the **Controller Agent**, enabling dynamic tool-calling behavior.



In [5]:
import ast, operator as op, os, re
from dataclasses import dataclass
from typing import Any, Dict, List

# ---------- Base ----------
@dataclass
class ToolResult:
    content: str
    meta: Dict[str, Any] | None = None

class BaseTool:
    name: str = "base"
    description: str = ""
    def run(self, query: str) -> ToolResult:
        raise NotImplementedError

# ---------- Calculator Tool ----------
_ALLOWED_BINOPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
                   ast.Div: op.truediv, ast.FloorDiv: op.floordiv,
                   ast.Mod: op.mod, ast.Pow: op.pow}
_ALLOWED_UNARYOPS = {ast.UAdd: op.pos, ast.USub: op.neg}

def _eval_ast(node: ast.AST):
    if isinstance(node, ast.Expression): return _eval_ast(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_BINOPS[type(node.op)](_eval_ast(node.left), _eval_ast(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_UNARYOPS[type(node.op)](_eval_ast(node.operand))
    raise ValueError(f"Unsupported: {ast.dump(node)}")

def safe_calc(expr: str):
    expr = expr.replace("^","**")
    return _eval_ast(ast.parse(expr, mode="eval"))

class CalculatorTool(BaseTool):
    name, description = "calculator", "Safe arithmetic evaluator"
    def run(self, query: str) -> ToolResult:
        try:
            return ToolResult(str(safe_calc(query.strip())), {"expr": query})
        except Exception:
            tokens = re.findall(r"[0-9().+/*%^\-]+", query)
            return ToolResult(str(safe_calc("".join(tokens))), {"expr": "".join(tokens)})

# ---------- Web Search Tool ----------
class WebSearchTool(BaseTool):
    name, description = "web_search", "DuckDuckGo or Tavily web search"
    def __init__(self, max_results=5): self.max_results = max_results

    def _search_duckduckgo(self, query: str) -> List[dict]:
        from ddgs import DDGS

        with DDGS() as ddgs:
            return list(ddgs.text(query, max_results=self.max_results))

    def _search_tavily(self, query: str) -> List[dict]:
        from tavily import TavilyClient
        client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
        res = client.search(query)
        return [{"title": r["title"], "href": r["url"], "body": r["content"]}
                for r in res.get("results", [])][:self.max_results]

    def run(self, query: str) -> ToolResult:
        results = (self._search_tavily(query) if os.getenv("TAVILY_API_KEY")
                   else self._search_duckduckgo(query))
        if not results:
            return ToolResult("No results", {"query": query})
        top = results[0]
        summary = (f"{top.get('title','')}\n"
                   f"{top.get('body') or top.get('snippet','')}\n"
                   f"{top.get('href') or top.get('url','')}")
        return ToolResult(summary.strip(), {"query": query, "results": results})

## 2. Retrieval-Augmented Generation (RAG)

This section defines a **RAGTool** that combines:

- **SentenceTransformers (all-MiniLM-L6-v2)** for fast embeddings.
- **FAISS** for efficient vector similarity search.
- **Gemini (via google-generativeai)** for natural language answer generation.

The RAG pipeline works as:
1. Upload a document (`.pdf`, `.txt`, `.docx`).
2. The document is **chunked** into overlapping text passages.
3. Each chunk is **encoded** into embeddings and indexed with FAISS.
4. On user query, the top-k most relevant chunks are retrieved.
5. Gemini generates a **concise answer**, restricted to retrieved context.
6. Answer is returned with citation of the top source.





In [11]:
pip install sentence-transformers faiss-cpu


In [12]:
import os, re
from dataclasses import dataclass
from typing import List, Tuple

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import google.generativeai as genai

try:
    import pypdf   # PDF support
    _HAS_PDF=True
except:
    _HAS_PDF=False

try:
    import docx    # DOCX support
    _HAS_DOCX=True
except:
    _HAS_DOCX=False


@dataclass
class RetrievedChunk:
    source: str
    chunk_id: int
    text: str
    score: float


class RAGTool:
    """
    Retrieval-Augmented Generation using SentenceTransformers + FAISS + Gemini.
    Supports TXT, PDF, and DOCX.
    """
    def __init__(self, model_name="all-MiniLM-L6-v2", chunk_size=800, chunk_overlap=200, llm_name="gemini-1.5-pro"):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)

        self._chunks: List[Tuple[str, int, str]] = []
        self._index = None
        self._embeddings = None

        # Gemini LLM
        self.llm = genai.GenerativeModel(llm_name)

    def _clean(self, text: str) -> str:
        return re.sub(r"\s+", " ", text).strip()

    def _chunk(self, text: str) -> List[str]:
        out=[]; i=0; n=len(text)
        while i<n:
            out.append(text[i:i+self.chunk_size])
            i+=max(1, self.chunk_size-self.chunk_overlap)
        return out

    def load_file(self, path: str):
        ext = os.path.splitext(path.lower())[1]
        if ext==".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()
        elif ext==".pdf":
            if not _HAS_PDF: raise RuntimeError("pypdf not installed; cannot read PDFs.")
            reader = pypdf.PdfReader(path)
            text = "\n".join([(p.extract_text() or "") for p in reader.pages])
        elif ext==".docx":
            if not _HAS_DOCX: raise RuntimeError("python-docx not installed; cannot read DOCX.")
            document = docx.Document(path)
            text = "\n".join([p.text for p in document.paragraphs])
        else:
            raise ValueError(f"Unsupported file type: {ext}")

        text = self._clean(text)
        for cid, ck in enumerate(self._chunk(text)):
            self._chunks.append((os.path.basename(path), cid, ck))

    def build(self):
        if not self._chunks:
            raise RuntimeError("No documents loaded.")
        texts = [c[2] for c in self._chunks]
        self._embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

        dim = self._embeddings.shape[1]
        self._index = faiss.IndexFlatL2(dim)
        self._index.add(self._embeddings)

    def retrieve(self, query: str, k: int = 5) -> List[RetrievedChunk]:
        if self._index is None:
            raise RuntimeError("Index not built. Call build() first.")
        q_emb = self.model.encode([query], convert_to_numpy=True)
        D, I = self._index.search(q_emb, k)
        res=[]
        for rank, idx in enumerate(I[0]):
            src, cid, txt = self._chunks[idx]
            res.append(RetrievedChunk(src, cid, txt, float(D[0][rank])))
        return res

    def run(self, query: str, k: int = 5) -> str:
        # Step 1: Retrieve top chunks
        hits = self.retrieve(query, k=k)
        if not hits:
            return "No relevant passages found."

        # Step 2: Build context
        context = "\n\n".join([h.text for h in hits])

        # Step 3: Ask Gemini
        prompt = f"""
        You are a helpful assistant. Answer the question based ONLY on the provided context.

        Question: {query}

        Context:
        {context}

        Answer clearly and concisely:
        """
        try:
            response = self.llm.generate_content(prompt)
            answer = response.text.strip()

            # Step 4: Return answer + cite first source
            first_source = hits[0].source
            return f"{answer}\n\n(Source: {first_source}, top {len(hits)} chunks used)"
        except Exception as e:
            return f"[RAG Error: {e}]"


In [13]:
!pip install pypdf python-docx scikit-learn




## 3. Hybrid Math Word Problem Solver (Gemini + Calculator)

This module implements a **Hybrid Math Solver** that combines:
- **Gemini (`gemini-1.5-pro`)** → for step-by-step reasoning in natural language.  
- **CalculatorTool** → for exact arithmetic evaluation of the extracted numeric expression.  

### Workflow
1. **Query Handling** → A math word problem is passed to Gemini.  
2. **Reasoning** → Gemini generates a step-by-step solution and explicitly marks the answer with `"Final Answer: X"`.  
3. **Expression Extraction** → The solver attempts to parse the final numeric expression or number from the reasoning.  
4. **Verification** → The expression is re-evaluated using the `CalculatorTool` to ensure correctness.  

### Evaluation
To validate solver performance, we benchmark against the **GSM8k dataset** (grade-school math word problems).  

- A helper function extracts the numeric "gold answer" from dataset annotations.  
- Predictions are compared against gold answers for a subset of 50 problems.  
- Accuracy is reported, along with sample Q/A outputs for inspection.  




In [14]:
import re
import google.generativeai as genai

class HybridMathWordProblemSolverTool(BaseTool):
    name, description = "math_word_solver", "Solves math word problems (GSM8k style) using Gemini + Calculator"

    def __init__(self, model_name="gemini-1.5-pro"):
        self.model = genai.GenerativeModel(model_name)
        self.calc = CalculatorTool()  # reuse your calculator tool

    def _extract_last_number_or_expr(self, text: str):
        """
        Extract the final numeric answer from Gemini's output.
        Priority:
        1. Lines starting with 'Answer', 'Final', or 'Therefore'
        2. The last standalone number in the text
        """
        lines = text.strip().splitlines()

        # 1. Look for "Answer: X" or "Final Answer: X"
        for line in reversed(lines):  # bottom-up
            match = re.search(r"(?:Answer|Final|Therefore)[:\s]+(-?\d+\.?\d*)", line, re.I)
            if match:
                return match.group(1)

        # 2. Otherwise, take the last number in the text
        nums = re.findall(r"-?\d+\.?\d*", text)
        if nums:
            return nums[-1]

        return None

    def run(self, query: str) -> ToolResult:
        prompt = f"""
        You are a math tutor. Solve this problem step by step.
        At the end, write explicitly: "Final Answer: X"

        Problem: {query}
        """
        try:
            response = self.model.generate_content(prompt)
            reasoning = response.text.strip()

            expr = self._extract_last_number_or_expr(reasoning)
            if expr:
                try:
                    calc_result = self.calc.run(expr).content
                    return ToolResult(calc_result, {
                        "model": "gemini",
                        "reasoning": reasoning,
                        "calc_checked": expr
                    })
                except Exception:
                    return ToolResult(expr, {
                        "model": "gemini",
                        "reasoning": reasoning,
                        "calc_checked": "failed"
                    })
            return ToolResult(reasoning, {"model": "gemini", "note": "no numeric found"})

        except Exception as e:
            return ToolResult(f"Error: {e}", {"query": query})


# === Evaluation helpers ===
def extract_final_number(answer: str):
    """Extract the final numeric answer (after #### if available)."""
    if "####" in answer:
        answer = answer.split("####")[-1].strip()
    nums = re.findall(r"-?\d+\.?\d*", answer)
    if not nums:
        return None
    return int(nums[-1]) if nums[-1].isdigit() else float(nums[-1])

def evaluate_math_solver(tool, dataset, n=50):
    correct = 0
    total = 0
    results = []

    subset = dataset["train"].select(range(n))  # select first n samples

    for sample in subset:
        q = sample["question"]
        gold = extract_final_number(sample["answer"])
        pred_raw = tool.run(q).content.strip()
        pred = extract_final_number(pred_raw)

        total += 1
        if gold is not None and pred == gold:
            correct += 1

        results.append((q, gold, pred, pred_raw))

    accuracy = correct / total * 100
    return accuracy, results


In [15]:
math_tool = HybridMathWordProblemSolverTool()

q1 = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did she sell altogether in April and May?"
print("Q:", q1)
print("A:", math_tool.run(q1).content)

q2 = "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents gave her $15, and her grandparents twice as much. How much more money does Betty need?"
print("Q:", q2)
print("A:", math_tool.run(q2).content)


Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did she sell altogether in April and May?
A: 72
Q: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents gave her $15, and her grandparents twice as much. How much more money does Betty need?
A: 5


### 4. Benchmarking the Hybrid Math Solver on GSM8k

To rigorously evaluate solver performance, we used the **GSM8k dataset**  
(grade-school math word problems, curated by OpenAI).

#### Setup
- Load the dataset via Hugging Face (`openai/gsm8k`, "main" split).  
- Run the solver on the first 50 training samples.  
- Extract gold answers and compare against model predictions.  
- Compute overall accuracy.  

#### Results
The solver achieved **96% accuracy** on 50 GSM8k problems.

Example outputs:


In [22]:
from datasets import load_dataset

# Load GSM8k dataset
gsm8k = load_dataset("openai/gsm8k", "main")

# Now evaluate
math_tool = HybridMathWordProblemSolverTool()
acc, samples = evaluate_math_solver(math_tool, gsm8k, n=50)

print(f"Accuracy on 50 GSM8k problems: {acc:.2f}%")
for q, gold, pred, pred_raw in samples[:10]:
    print("Q:", q)
    print("Gold:", gold)
    print("Pred:", pred)
    print("Raw Output:", pred_raw)
    print("---")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Accuracy on 50 GSM8k problems: 98.00%
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Gold: 72
Pred: 72
Raw Output: 72
---
Q: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Gold: 10
Pred: 10
Raw Output: 10
---
Q: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
Gold: 5
Pred: 5
Raw Output: 5
---
Q: Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?
Gold: 42
Pred: 42
Raw Output: 42
---
Q: James writes a 3-page letter to 2 different friends

# 5. ControllerAgent: Multi-Tool Routing AI


The agent can handle queries using multiple tools:

1. **Calculator (`calculator`)**  
   - Handles direct arithmetic expressions, e.g., `2+2` or `1000/25`.

2. **Math Word Problem Solver (`math_word_solver`)**  
   - Solves word problems, e.g., "Betty has 100 dollars, spends 25, how much is left?"

3. **RAG (`rag`)**  
   - Retrieves answers from uploaded documents.  
   - Requires files to be loaded via `rag_tool.load_file()` and indexed via `rag_tool.build()`.

4. **Web Search (`web_search`)**  
   - Handles current events, factual lookups, news, weather, or general online information.

5. **Fallback / General Chat (`fallback`)**  
   - Handles chit-chat or unsupported queries via Gemini LLM.

### **How it Works**

1. **Query Classification**  
   - Uses Gemini to classify the query and choose the best tool from the list above.

2. **Tool Execution**  
   - Once classified, the agent calls the corresponding tool's `run()` method.  
   - Errors (e.g., no document loaded, network issues) are caught and displayed gracefully.

3. **RAG Behavior**  
   - If multiple documents are uploaded, retrieval is based on semantic similarity.  
   - Queries may return content from the most relevant document chunks.

4. **Fallback Handling**  
   - Queries not matched to any specific tool are answered directly by Gemini.


In [16]:
import google.generativeai as genai

class ControllerAgent:
    """
    Routes user queries to the appropriate tool:
    - Calculator for direct arithmetic
    - Math solver for word problems
    - RAG for doc-based Q&A
    - Web search for external queries
    - Gemini fallback for chit-chat/general
    """
    def __init__(self, tools: dict, llm_name="gemini-1.5-pro"):
        self.tools = tools  # dict of tool_name -> tool instance
        self.llm = genai.GenerativeModel(llm_name)

    def route(self, query: str) -> str:
        """
        Use Gemini to classify intent, then call the right tool.
        """
        classification_prompt = f"""
        You are an AI router. Decide which TOOL should answer the query.

        Available tools:
        - calculator: Direct arithmetic (e.g., "2+2", "1000/25").
        - math_word_solver: Word problems (e.g., "If John has 2 apples and eats 1...").
        - rag: Question about uploaded document.
        - web_search: Current events, factual lookup, weather, news, or the internet.
        - fallback: General chit-chat or unsupported.

        Query: {query}

        Respond with ONLY the tool name from [calculator, math_word_solver, rag, web_search, fallback].
        """
        try:
            resp = self.llm.generate_content(classification_prompt)
            tool_choice = resp.text.strip().lower()
        except Exception as e:
            return f"[Routing Error: {e}]"

        if tool_choice not in self.tools and tool_choice != "fallback":
            tool_choice = "fallback"

        # --- Handle fallback (general chit-chat) ---
        if tool_choice == "fallback":
            try:
                return f"💡 Gemini: {self.llm.generate_content(query).text.strip()}"
            except Exception as e:
                return f"[Gemini Error: {e}]"

        # --- Handle RAG safely ---
        if tool_choice == "rag":
            try:
                result = self.tools["rag"].run(query)
                return f"🛠️ Tool [rag]: {result}"
            except RuntimeError:
                return "⚠️ No document is loaded yet. Please upload and build the index first."
            except Exception as e:
                return f"[RAG Error: {e}]"

        # --- Handle all other tools ---
        try:
            tool = self.tools[tool_choice]
            result = tool.run(query)
            return f"🛠️ Tool [{tool_choice}]: {result.content}"
        except Exception as e:
            return f"[{tool_choice} Error: {e}]"


# === Initialize all tools ===
calc_tool = CalculatorTool()
web_tool = WebSearchTool()
rag_tool = RAGTool()   # (remember: you need to load_file + build() before using)
math_tool = HybridMathWordProblemSolverTool()

tools = {
    "calculator": calc_tool,
    "web_search": web_tool,
    "rag": rag_tool,
    "math_word_solver": math_tool,
}

agent = ControllerAgent(tools)

# === Example runs ===
print(agent.route("What is 25*40?"))               # Calculator
print(agent.route("Betty has 100 dollars, spends 25, how much left?"))  # Math solver
print(agent.route("What is the weather in Topi today?"))  # Web search
print(agent.route("Summarize the uploaded document"))     # RAG
print(agent.route("Tell me a joke"))                      # Fallback chit-chat


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🛠️ Tool [calculator]: 1000
🛠️ Tool [math_word_solver]: 75
🛠️ Tool [web_search]: Topi, Khyber Pakhtunkhwa, Pakistan Weather Forecast | AccuWeather
Topi , Khyber Pakhtunkhwa, Pakistan Weather Forecast, with current conditions, wind, air quality, and what to expect for the next 3 days.
https://www.accuweather.com/en/pk/topi/259286/weather-forecast/259286
⚠️ No document is loaded yet. Please upload and build the index first.
💡 Gemini: Why don't scientists trust atoms? 

Because they make up everything!


# 6. Multi-Document RAG Chat Interface

This cell implements an **interactive chat interface** that allows you to ask questions about uploaded documents and interact with the multi-tool `ControllerAgent`.

### **Features**

1. **Multiple Document Uploads**
   - Upload `.pdf`, `.txt`, or `.docx` files.  
   - Each uploaded document is indexed via the RAG tool and can be queried.

2. **Interactive Chat**
   - User messages appear in **blue chat bubbles** on the right.  
   - Agent responses appear in **gray chat bubbles** on the left.

3. **Persistent RAG Context**
   - The agent remembers all uploaded documents and uses them for subsequent queries.  
   - RAG retrieval is based on semantic similarity to find the most relevant document chunks.

4. **Tool Routing**
   - Queries are automatically routed by the `ControllerAgent`:
     - `rag` for document-based queries  
     - `calculator` for arithmetic  
     - `math_word_solver` for word problems  
     - `web_search` for online queries  
     - `fallback` for chit-chat

5. **Auto-Scroll & Error Handling**
   - The chat automatically scrolls to the latest message.  
   - Upload errors and tool errors are displayed directly in the chat.

### **Usage Instructions**

1. **Upload documents** using the file uploader.  
2. Wait for the system to index the files.  
3. **Type your question** in the input box and press **Enter** or click **Send**. {make sure to be specfic with your question}
4. The agent responds using the uploaded documents and other tools as appropriate.


In [21]:
import ipywidgets as widgets
from IPython.display import display
import tempfile, os

# --- Chat Log Widget ---
chat_log = widgets.HTML(
    value="<div style='font-family:Arial; font-size:15px; line-height:1.5;'><b>💬 Chat Started...</b></div>",
    layout=widgets.Layout(height="400px", overflow_y="auto", border="1px solid #ddd", padding="10px", background_color="white")
)

# --- Chat Input ---
chat_input = widgets.Text(placeholder="Type your question...", layout=widgets.Layout(width="80%", padding="5px"))
send_button = widgets.Button(description="Send", button_style="primary", layout=widgets.Layout(width="18%"))

# --- File Upload ---
upload = widgets.FileUpload(accept='.pdf,.txt,.docx', multiple=True)
upload_status = widgets.HTML(value="<i style='color:gray'>📂 Upload documents for RAG...</i>")

# --- Track uploaded docs ---
uploaded_docs = []

# --- Helper: append message to chat with auto-scroll ---
def append_chat(user_text=None, agent_text=None, color_user="#0078D7", color_agent="#f1f0f0"):
    if user_text:
        chat_log.value += f"""
        <div style='text-align:right; margin:5px 0;'>
          <span style='display:inline-block; background:{color_user}; color:white; padding:8px 12px; border-radius:15px; max-width:70%;'>
            {user_text}
          </span>
        </div>
        """
    if agent_text:
        chat_log.value += f"""
        <div style='text-align:left; margin:5px 0;'>
          <span style='display:inline-block; background:{color_agent}; color:#333; padding:8px 12px; border-radius:15px; max-width:70%;'>
            {agent_text}
          </span>
        </div>
        """
    # Auto-scroll to bottom
    chat_log.value += "<script>this.parentNode.scrollTop = this.parentNode.scrollHeight;</script>"

# --- Upload Handling ---
def handle_upload(change):
    global rag_tool, uploaded_docs
    if upload.value:
        for item in upload.value.values():
            fname = item.get("metadata", {}).get("name") or item.get("name")
            content = item["content"]

            temp_path = os.path.join(tempfile.gettempdir(), fname)
            with open(temp_path, "wb") as f:
                f.write(content)

            try:
                # Load + build RAG for each document
                rag_tool.load_file(temp_path)
                uploaded_docs.append(fname)
            except Exception as e:
                append_chat(agent_text=f"❌ Error loading '{fname}': {e}")

        try:
            rag_tool.build()  # Build after all uploaded files
            upload_status.value = f"<span style='color:green; font-weight:bold;'>✅ Documents uploaded: {', '.join(uploaded_docs)}</span>"
            append_chat(agent_text=f"System: {', '.join(uploaded_docs)} are ready for querying.")
        except Exception as e:
            upload_status.value = f"<span style='color:red;'>❌ Error building index: {e}</span>"

upload.observe(handle_upload, names="value")

# --- Chat Handling ---
def handle_chat(_):
    query = chat_input.value.strip()
    if not query:
        return

    append_chat(user_text=query)
    chat_input.value = ""

    try:
        # Route query through agent
        reply = agent.route(query)  # Assuming agent internally uses rag_tool persistently
    except Exception as e:
        reply = f"⚠️ Error: {e}"

    append_chat(agent_text=reply)

send_button.on_click(handle_chat)
chat_input.on_submit(handle_chat)

# --- Layout ---
ui = widgets.VBox([
    widgets.HBox([upload, upload_status]),
    chat_log,
    widgets.HBox([chat_input, send_button])
])

display(ui)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]